Fase 5 Modul 2: JOIN, Subquery, CTE

In [1]:
from pathlib import Path
import pandas as pd, sqlite3
csv = Path("data/clean_tips.csv") if Path("data/clean_tips.csv").exists() else Path("../data/clean_tips.csv")
db = csv.parent / "tips.db"
con = sqlite3.connect(db)
# kamus hari: Inggris -> Indonesia + jenis. day = kunci, wajib unik.
day_info = pd.DataFrame({"day": ["Thur","Fri","Sat","Sun"],
    "hari_id": ["Kamis","Jumat","Sabtu","Minggu"], "jenis": ["Weekday","Weekday","Weekend","Weekend"]})
day_info.to_sql("day_info", con, if_exists="replace", index=False)
print(pd.read_sql("SELECT * FROM day_info", con))
print(pd.read_sql("SELECT day, COUNT(*) FROM day_info GROUP BY day HAVING COUNT(*)>1", con))  # harusnya kosong = kunci unik

    day hari_id    jenis
0  Thur   Kamis  Weekday
1   Fri   Jumat  Weekday
2   Sat   Sabtu  Weekend
3   Sun  Minggu  Weekend
Empty DataFrame
Columns: [day, COUNT(*)]
Index: []


In [2]:
# LEFT JOIN: semua struk tetap ada + tempelan kamus
q = """SELECT t.day, d.hari_id, t.total_bayar FROM tips t
LEFT JOIN day_info d ON t.day = d.day LIMIT 3"""
print(pd.read_sql(q, con))
# cek kembung: COUNT sesudah join harus = 241. Kalau meledak, kunci kanan duplikat.
print(pd.read_sql("SELECT COUNT(*) AS n FROM tips", con))
print(pd.read_sql("SELECT COUNT(*) AS n FROM tips t LEFT JOIN day_info d ON t.day=d.day", con))

   day hari_id  total_bayar
0  Sat   Sabtu        24.51
1  Sun  Minggu        26.99
2  Sun  Minggu        28.20
     n
0  241
     n
0  241


In [3]:
# Subquery: transaksi di atas rata-rata (rata-rata dihitung di dalam)
print(pd.read_sql("""SELECT day, total_bayar FROM tips
WHERE total_bayar > (SELECT AVG(total_bayar) FROM tips) LIMIT 3""", con))
# CTE: beri nama tiap langkah. Sama hasil, tapi tiap langkah bisa dites 1-1.
print(pd.read_sql("""WITH rata AS (SELECT AVG(total_bayar) AS v FROM tips),
atas AS (SELECT day, total_bayar FROM tips, rata WHERE total_bayar > rata.v)
SELECT COUNT(*) AS n, AVG(total_bayar) AS avg_atas FROM atas""", con))

   day  total_bayar
0  Sat        24.51
1  Sun        26.99
2  Sun        28.20
    n   avg_atas
0  99  32.183535
